<a href="https://colab.research.google.com/github/mhabibier/scikit-learn-cookbook/blob/main/01_Common_Conventions_and_API_Elements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 — Common Conventions and API Elements of scikit-learn

**Nama:** Muhammad Habibie Rabbani  
**Kelas:** TK-47-04  
**Program Studi:** Teknik Komputer  
**Universitas:** Telkom University  

**Primary Reference:**  
John Sukup, *scikit-learn Cookbook: Over 80 Recipes for Machine
Learning in Python with scikit-learn*, Third Edition, Packt
Publishing, 2025.

## Tujuan Pembelajaran

Setelah menyelesaikan chapter ini, pembaca diharapkan mampu:

1. Menjelaskan filosofi desain API scikit-learn.
2. Menjelaskan fungsi estimator dan transformer.
3. Menggunakan `fit()`, `predict()`, `transform()`, dan
   `fit_transform()`.
4. Menjelaskan custom estimator dan custom transformer.
5. Menjelaskan fungsi pipeline dalam workflow machine learning.
6. Mengakses atribut hasil training seperti `coef_` dan `intercept_`.
7. Mengatur hyperparameter menggunakan `set_params()` dan
   `get_params()`.
8. Menjelaskan GridSearchCV dan RandomizedSearchCV.
9. Menjelaskan estimator tags dan metadata routing.
10. Menerapkan praktik penggunaan API scikit-learn yang baik.

In [1]:
import sys
import numpy as np
import sklearn

RANDOM_STATE = 42

print("Python version       :", sys.version.split()[0])
print("NumPy version        :", np.__version__)
print("scikit-learn version :", sklearn.__version__)

Python version       : 3.12.14
NumPy version        : 2.3.5
scikit-learn version : 1.8.0


### Interpretasi Environment

Versi Python, NumPy, dan scikit-learn dicatat untuk mendukung
reproducibility. Perbedaan versi library dapat mengubah default parameter,
peringatan, atau output beberapa algoritma.

## 1. Introduction to scikit-learn's Design Philosophy

Desain scikit-learn dibangun berdasarkan empat prinsip utama:

1. **Consistency**  
   Sebagian besar estimator menggunakan pola API yang sama, seperti
   `fit()`, `predict()`, dan `transform()`.

2. **Simplicity**  
   Banyak algoritma mempunyai nilai parameter awal yang memungkinkan
   pengguna membuat model dasar dengan relatif sedikit kode.

3. **Modularity**  
   Estimator, transformer, dan pipeline dapat digabungkan menjadi workflow
   machine learning.

4. **Reusability**  
   Komponen yang sudah dibuat dapat digunakan kembali pada eksperimen atau
   dataset lain.

Konsistensi API membuat pengguna dapat mengganti algoritma tanpa harus
mengubah keseluruhan struktur program.

## 2. Understanding Estimators

Estimator adalah objek yang mempelajari parameter atau pola dari data.
Semua estimator memiliki metode `fit()`.

Pada supervised learning:

- `fit(X, y)` mempelajari hubungan antara feature dan target.
- `predict(X)` menghasilkan prediksi untuk data baru.
- `score(X, y)` menghitung metrik default estimator.

Parameter yang ditentukan sebelum training disebut hyperparameter.
Nilai yang dipelajari setelah training biasanya disimpan pada atribut
dengan akhiran underscore, seperti `coef_` dan `intercept_`.

In [2]:
from sklearn.linear_model import LinearRegression

X_regression = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
])

y_regression = np.array([
    1,
    2,
    3,
    3.5,
    5,
])

linear_model = LinearRegression()
linear_model.fit(
    X_regression,
    y_regression,
)

X_new = np.array([
    [6],
    [7],
])

predictions = linear_model.predict(
    X_new
)

print("Predictions:", predictions)

Predictions: [5.75 6.7 ]


### Analisis Linear Regression

`LinearRegression` mempelajari hubungan linear antara feature `X` dan
target `y`. Setelah `fit()` dijalankan, model memperoleh slope dan
intercept yang digunakan untuk memprediksi nilai baru.

Prediksi untuk `X = 6` adalah sekitar 5,75, sedangkan prediksi untuk
`X = 7` adalah sekitar 6,70. Hasil ini merupakan estimasi berdasarkan
garis regresi terbaik dari lima observasi training.

Contoh ini menggunakan dataset sangat kecil hanya untuk menunjukkan
keseragaman API estimator. Hasilnya tidak cukup untuk mewakili model
machine learning pada data nyata.

### `fit_predict()` pada KMeans

Pada clustering, `fit_predict(X)` menyesuaikan pusat cluster lalu menghasilkan
label untuk observasi yang sama. Angka label hanya penanda cluster, bukan
peringkat kualitas.


In [3]:
from sklearn.cluster import KMeans

X_clustering = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
])

kmeans = KMeans(
    n_clusters=2,
    random_state=RANDOM_STATE,
    n_init=10,
)

cluster_labels = kmeans.fit_predict(
    X_clustering
)

print("Cluster labels :", cluster_labels)
print(
    "Cluster centers:",
    kmeans.cluster_centers_.flatten(),
)

Cluster labels : [0 0 0 1 1]
Cluster centers: [2.  4.5]


### Analisis KMeans

KMeans membagi lima observasi menjadi dua kelompok berdasarkan kedekatan
nilainya. Nilai kecil akan berada pada satu cluster, sedangkan nilai lebih
besar berada pada cluster lainnya.

Nomor label cluster dapat berbeda, misalnya `[1, 1, 1, 0, 0]` atau
`[0, 0, 0, 1, 1]`. Hal ini tidak menunjukkan hasil yang berbeda karena
nomor cluster hanya merupakan identifier. Hal yang penting adalah anggota
setiap cluster.

`random_state=42` dan `n_init=10` digunakan agar proses inisialisasi
cluster dapat direproduksi.

## 3. Transformers and the `transform()` Method

Transformer adalah estimator yang mengubah bentuk atau skala data.

Transformer umumnya menggunakan:

- `fit(X)` untuk mempelajari parameter transformasi.
- `transform(X)` untuk menerapkan parameter tersebut.
- `fit_transform(X)` untuk mempelajari dan langsung mentransformasikan
  data.

`StandardScaler` menghitung mean dan standard deviation setiap feature,
kemudian melakukan standardisasi menggunakan:

$$
z = \frac{x-\mu}{\sigma}
$$

Hasil standardisasi mempunyai mean mendekati nol dan standard deviation
mendekati satu.

In [4]:
from sklearn.preprocessing import StandardScaler

X_scaling = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
])

scaler = StandardScaler()

scaler.fit(
    X_scaling
)

X_scaled = scaler.transform(
    X_scaling
)

print("Original data:")
print(X_scaling)

print("\nScaled data:")
print(X_scaled)

print("\nLearned mean:")
print(scaler.mean_)

print("\nLearned scale:")
print(scaler.scale_)

Original data:
[[1 2]
 [3 4]
 [5 6]]

Scaled data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Learned mean:
[3. 4.]

Learned scale:
[1.63299316 1.63299316]


### Analisis StandardScaler

`fit()` menghitung mean dan standard deviation dari setiap kolom.
`transform()` menggunakan parameter yang sudah dipelajari untuk
mengubah data.

Baris tengah menghasilkan nilai nol karena nilainya sama dengan mean
setiap feature. Baris pertama menghasilkan nilai negatif dan baris
terakhir menghasilkan nilai positif.

Pada workflow sebenarnya, `fit()` hanya boleh dijalankan menggunakan
data training. Menjalankan `fit()` pada data test dapat menyebabkan
data leakage.

In [5]:
scaler_fit_transform = StandardScaler()

X_scaled_directly = (
    scaler_fit_transform.fit_transform(
        X_scaling
    )
)

print(X_scaled_directly)

print(
    "\nColumn means:",
    X_scaled_directly.mean(axis=0),
)

print(
    "Column standard deviations:",
    X_scaled_directly.std(axis=0),
)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Column means: [0. 0.]
Column standard deviations: [1. 1.]


### Analisis `fit_transform()`

`fit_transform()` menghasilkan transformasi yang sama dengan menjalankan
`fit()` kemudian `transform()` pada data yang sama.

Metode ini biasanya digunakan pada data training. Data test hanya
menggunakan `transform()` dari scaler yang telah dipelajari menggunakan
data training.

## 4. Handling Custom Estimators and Transformers

Custom estimator memungkinkan pengguna membuat algoritma atau
transformasi sendiri yang tetap kompatibel dengan scikit-learn.

Custom transformer biasanya mewarisi:

- `BaseEstimator`, yang menyediakan pengelolaan parameter.
- `TransformerMixin`, yang menyediakan implementasi dasar
  `fit_transform()`.

Custom transformer setidaknya mengimplementasikan:

- `fit(self, X, y=None)`, yang mengembalikan `self`.
- `transform(self, X)`, yang menghasilkan data hasil transformasi.

Atribut yang dipelajari pada `fit()` sebaiknya memiliki akhiran
underscore. Sebelum transformasi, `check_is_fitted()` dapat digunakan
untuk memastikan bahwa transformer sudah dilatih.

In [6]:
from sklearn.base import (
    BaseEstimator,
    TransformerMixin,
)
from sklearn.utils.validation import (
    check_array,
    check_is_fitted,
)


class ValueClipper(
    TransformerMixin,
    BaseEstimator,
):
    """Limit numerical values to a selected range."""

    def __init__(
        self,
        lower=0.0,
        upper=1.0,
    ):
        self.lower = lower
        self.upper = upper

    def fit(
        self,
        X,
        y=None,
    ):
        X_checked = check_array(X)

        self.n_features_in_ = (
            X_checked.shape[1]
        )

        self.is_fitted_ = True

        return self

    def transform(
        self,
        X,
    ):
        check_is_fitted(
            self,
            "is_fitted_",
        )

        X_checked = check_array(X)

        return np.clip(
            X_checked,
            self.lower,
            self.upper,
        )


X_custom = np.array([
    [-0.4, 0.5],
    [0.8, 1.4],
    [2.0, -1.0],
])

clipper = ValueClipper(
    lower=0.0,
    upper=1.0,
)

X_clipped = clipper.fit_transform(
    X_custom
)

print("Original data:")
print(X_custom)

print("\nClipped data:")
print(X_clipped)

print("\nTransformer parameters:")
print(clipper.get_params())

Original data:
[[-0.4  0.5]
 [ 0.8  1.4]
 [ 2.  -1. ]]

Clipped data:
[[0.  0.5]
 [0.8 1. ]
 [1.  0. ]]

Transformer parameters:
{'lower': 0.0, 'upper': 1.0}


### Analisis Custom Transformer

`ValueClipper` membatasi tiap nilai ke interval 0 sampai 1. Nilai -0,4
menjadi 0 dan 1,4 menjadi 1. `fit()` mencatat jumlah feature dan
`transform()` memeriksa bahwa objek sudah dilatih sebelum mengubah data.
Aturan clipping harus ditentukan berdasarkan kebutuhan data; nilai ekstrem
tidak selalu salah dan membatasi nilainya dapat menghilangkan informasi.


## 5. Pipelines and Workflow Automation

Pipeline menggabungkan beberapa proses menjadi satu objek. Transformer
ditempatkan pada langkah awal, sedangkan estimator berada pada langkah
terakhir.

Saat `fit()` dijalankan:

1. Pipeline menjalankan `fit_transform()` pada transformer.
2. Hasil transformasi dikirim ke langkah berikutnya.
3. Estimator terakhir menjalankan `fit()`.

Saat `predict()` dijalankan:

1. Data baru melewati seluruh transformer menggunakan `transform()`.
2. Estimator terakhir menghasilkan prediksi.

Pipeline membantu menjaga urutan proses, meningkatkan reproducibility,
dan mengurangi risiko data leakage.

In [7]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

iris = load_iris(
    as_frame=True
)

X_iris = iris.data
y_iris = iris.target

X_train, X_test, y_train, y_test = (
    train_test_split(
        X_iris,
        y_iris,
        test_size=0.20,
        stratify=y_iris,
        random_state=RANDOM_STATE,
    )
)

pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipeline.fit(
    X_train,
    y_train,
)

y_pred = pipeline.predict(
    X_test
)

test_accuracy = accuracy_score(
    y_test,
    y_pred,
)

print(
    f"Training size : {len(X_train)}"
)
print(
    f"Testing size  : {len(X_test)}"
)
print(
    f"Test accuracy : {test_accuracy:.4f}"
)

Training size : 120
Testing size  : 30
Test accuracy : 0.9333


### Analisis Pipeline

Pipeline menggabungkan `StandardScaler` dan `LogisticRegression`.
Scaler hanya mempelajari parameter dari data training karena seluruh
proses dilakukan di dalam pipeline setelah train-test split.

Penggunaan `stratify=y_iris` menjaga proporsi kelas pada data training
dan testing. Akurasi test menunjukkan proporsi prediksi yang benar pada
data yang tidak digunakan untuk training.

Nilai akurasi yang tinggi pada dataset Iris tidak otomatis menunjukkan
model akan memberikan performa yang sama pada dataset lain.

### Kaitan Pipeline dan MLOps

Pipeline menyimpan urutan preprocessing dan estimator dalam satu objek yang
dapat dijalankan berulang. Pada sistem produksi, pekerjaan tambahan mencakup
versi data dan model, pengujian, pemantauan perubahan data dan performa,
serta keputusan kapan model perlu dilatih ulang.


## 6. Common Attributes and Methods

Setelah `fit()` dijalankan, estimator menyimpan parameter yang telah
dipelajari. Atribut hasil training menggunakan akhiran underscore.

Contoh:

- `coef_`: koefisien model.
- `intercept_`: intercept atau bias.
- `classes_`: daftar kelas pada classifier.
- `cluster_centers_`: pusat cluster pada KMeans.

Metode `score()` memberikan metrik default. Untuk regression, metrik
default umumnya R-squared. Untuk classification, metrik default umumnya
accuracy.

In [8]:
print(
    "Coefficient:",
    linear_model.coef_,
)

print(
    "Intercept:",
    linear_model.intercept_,
)

print(
    "R-squared:",
    linear_model.score(
        X_regression,
        y_regression,
    ),
)

Coefficient: [0.95]
Intercept: 0.04999999999999938
R-squared: 0.9809782608695652


### Analisis Atribut Model

Koefisien sekitar 0,95 menunjukkan bahwa peningkatan satu unit `X`
berhubungan dengan peningkatan prediksi target sekitar 0,95.

Intercept sekitar 0,05 merupakan prediksi ketika `X` bernilai nol.
R-squared sekitar 0,981 menunjukkan bahwa model menjelaskan sebagian
besar variasi target pada data contoh.

Nilai tersebut dihitung pada data training sehingga belum menunjukkan
kemampuan generalisasi model pada data baru.

## 7. Hyperparameter Tuning with Search Methods

Hyperparameter adalah konfigurasi model yang ditentukan sebelum
training. Hyperparameter berbeda dari parameter model karena parameter
dipelajari dari data.

scikit-learn menyediakan:

- `get_params()` untuk membaca hyperparameter.
- `set_params()` untuk mengubah hyperparameter.
- `GridSearchCV()` untuk mencoba seluruh kombinasi dalam grid.
- `RandomizedSearchCV()` untuk mencoba sejumlah kombinasi acak.

In [9]:
from sklearn.ensemble import (
    RandomForestClassifier,
)

random_forest = (
    RandomForestClassifier()
)

random_forest.set_params(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE,
)

selected_parameters = {
    key: value
    for key, value
    in random_forest.get_params().items()
    if key in {
        "n_estimators",
        "max_depth",
        "random_state",
    }
}

print(selected_parameters)

{'max_depth': 10, 'n_estimators': 100, 'random_state': 42}


### Analisis Parameter Management

`set_params()` mengubah konfigurasi estimator tanpa membuat objek baru.
`get_params()` mengembalikan seluruh konfigurasi estimator.

Pada contoh ini, random forest akan menggunakan 100 decision tree,
kedalaman maksimum 10, dan random seed 42. Model belum dilatih karena
metode `fit()` belum dijalankan.

In [10]:
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)

parameter_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
    ]
}

cross_validation = (
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    scoring="accuracy",
    cv=cross_validation,
    n_jobs=-1,
)

grid_search.fit(
    X_train,
    y_train,
)

print(
    "Best parameters:",
    grid_search.best_params_,
)

print(
    "Best CV accuracy:",
    round(
        grid_search.best_score_,
        4,
    ),
)

print(
    "Test accuracy:",
    round(
        grid_search.score(
            X_test,
            y_test,
        ),
        4,
    ),
)

Best parameters: {'classifier__C': 10.0}
Best CV accuracy: 0.9667
Test accuracy: 1.0


### Analisis GridSearchCV

`GridSearchCV` mencoba empat nilai `C` untuk classifier dalam pipeline.
Awalan `classifier__` menunjuk langkah bernama `classifier`. Setiap
kombinasi dievaluasi dengan lima lipatan stratified pada data training.
`best_score_` adalah rerata akurasi cross-validation, sedangkan skor pada
`X_test` adalah evaluasi data yang tidak dipakai dalam pemilihan `C`.
Perbedaan dua skor wajar karena pembagian sampel berbeda; hasil test pada
Iris tetap perlu ditafsirkan sesuai jumlah data dan ruang lingkup dataset.


## 8. Working with Metadata: Tags and More

Estimator tags menjelaskan kemampuan sebuah estimator, misalnya:

- Apakah estimator membutuhkan proses `fit()`.
- Apakah estimator mendukung multi-output.
- Apakah estimator menerima missing value.
- Apakah proses estimator bersifat nondeterministic.

Metadata routing mengatur pengiriman informasi tambahan seperti:

- `sample_weight`.
- Group labels.
- Parameter tambahan untuk `fit()`.
- Informasi yang dibutuhkan scorer.

Router meneruskan metadata, sedangkan consumer menggunakan metadata
tersebut. Sebagai contoh, `sample_weight` dapat dikirim ke classifier
tanpa harus diterapkan pada scaler.

## 9. Best Practices for API Usage

Praktik yang digunakan dalam notebook ini meliputi:

1. Menggunakan API `fit()`, `predict()`, dan `transform()` secara
   konsisten.
2. Menggunakan nama variabel berbeda untuk setiap eksperimen.
3. Menetapkan `random_state` pada proses acak.
4. Memisahkan data training dan testing sebelum preprocessing.
5. Menempatkan preprocessing di dalam pipeline.
6. Tidak mempelajari parameter preprocessing dari data test.
7. Melakukan hyperparameter tuning hanya pada data training.
8. Menggunakan cross-validation untuk mengestimasi kemampuan
   generalisasi.
9. Menampilkan versi library.
10. Memberikan interpretasi terhadap setiap output.

## 10. Ringkasan Metode

| Konsep | API utama | Fungsi |
|---|---|---|
| Estimator | `fit()` | Mempelajari parameter dari data |
| Predictor | `predict()` | Menghasilkan prediksi |
| Transformer | `transform()` | Mengubah data |
| Fit-transform | `fit_transform()` | Mempelajari dan mengubah data |
| Unsupervised estimator | `fit_predict()` | Melatih dan menghasilkan label |
| Pipeline | `Pipeline()` | Menggabungkan preprocessing dan model |
| Model attributes | `coef_`, `intercept_` | Menyimpan parameter hasil training |
| Evaluation | `score()` | Menghasilkan metrik default |
| Parameter management | `get_params()`, `set_params()` | Membaca dan mengubah hyperparameter |
| Grid search | `GridSearchCV()` | Mencari kombinasi hyperparameter |
| Metadata routing | Router dan consumer | Mengarahkan metadata antar-komponen |

## 11. Kesimpulan

Chapter ini memperkenalkan konsistensi API scikit-learn melalui estimator,
transformer, dan pipeline. Estimator menggunakan `fit()` untuk mempelajari
pola dari data dan `predict()` untuk menghasilkan prediksi. Transformer
menggunakan `fit()` untuk mempelajari parameter transformasi dan
`transform()` untuk menerapkannya.

Pipeline menggabungkan preprocessing dan estimator dalam satu workflow.
Pendekatan ini membantu memastikan bahwa proses yang sama diterapkan pada
data training dan testing serta mengurangi risiko data leakage.

Atribut dengan akhiran underscore menyimpan informasi yang dipelajari
selama training. Sementara itu, hyperparameter ditentukan sebelum training
dan dapat dikelola melalui `get_params()`, `set_params()`, serta metode
search berbasis cross-validation.

Contoh dalam chapter ini menggunakan data sederhana untuk menjelaskan API.
Hasilnya tidak dimaksudkan sebagai model production. Pada data nyata,
kualitas data, pembagian data, pemilihan metrik, validasi, dan monitoring
harus dipertimbangkan secara lebih mendalam.

## 12. References

1. J. Sukup, *scikit-learn Cookbook: Over 80 Recipes for Machine
   Learning in Python with scikit-learn*, 3rd ed. Birmingham, UK:
   Packt Publishing, 2025.

2. scikit-learn Developers, “scikit-learn User Guide.”
   https://scikit-learn.org/stable/user_guide.html

3. Packt Publishing, “scikit-learn Cookbook, Third Edition:
   Code Repository.”
   https://github.com/PacktPublishing/scikit-learn-Cookbook-Third-Edition